In [3]:
import pandas as pd

# Load dataset and remove rows with null values
df = pd.read_csv("./datasets/ScamWatch.csv")
df.dropna(inplace=True)
df.head()

,StartOfMonth,Address_State,Scam___Contact_Mode,Complainant_Age,Complainant_Gender,Category_Level_2,Category_Level_3,Amount_lost,Number_of_reports
0,1/01/2025,Victoria,Email,65 and over,Male,Buying or selling,False billing,$69.00,41
1,1/02/2025,Victoria,Email,65 and over,Male,Buying or selling,False billing,$0.00,29
2,1/03/2025,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$2,643.00",39
3,1/04/2025,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$20,000.00",30
4,1/05/2025,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$86,416.00",28


In [4]:
# StartOfMonth: Split year and month 
df["StartOfMonth"] = pd.to_datetime(df["StartOfMonth"], dayfirst=True)

df["Year"] = df["StartOfMonth"].dt.year
df["Month"] = df["StartOfMonth"].dt.month

df.head()

,StartOfMonth,Address_State,Scam___Contact_Mode,Complainant_Age,Complainant_Gender,Category_Level_2,Category_Level_3,Amount_lost,Number_of_reports,Year,Month
0,2025-01-01,Victoria,Email,65 and over,Male,Buying or selling,False billing,$69.00,41,2025,1
1,2025-02-01,Victoria,Email,65 and over,Male,Buying or selling,False billing,$0.00,29,2025,2
2,2025-03-01,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$2,643.00",39,2025,3
3,2025-04-01,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$20,000.00",30,2025,4
4,2025-05-01,Victoria,Email,65 and over,Male,Buying or selling,False billing,"$86,416.00",28,2025,5


In [5]:
# Address_State: Extract seven states
states = ["Victoria", "New South Wales", "Queensland", "South Australia", "Tasmania", "Western Australia", "Northern Territory"]
df = df[df["Address_State"].isin(states)]

df["Address_State"].unique()

array(['Victoria', 'New South Wales', 'Queensland', 'Western Australia',
       'South Australia', 'Tasmania', 'Northern Territory'], dtype=object)

In [6]:
# Scam__Contact_Mode: Rename column and extract relevant rows
df.rename(columns={'Scam___Contact_Mode': 'Contact_Mode'}, inplace=True) # remove duplicate "_" by renaming column

contact_mode = ["Email", "Fax", "Phone call", "Text message", "Social media/Online forums", "In person"]
df = df[df["Contact_Mode"].isin(contact_mode)]

# Replace values
value_mapping = {
    "Phone call": "Phone",
    "Text message": "SMS",
    "Social media/Online forums": "Social media apps"
}

df["Contact_Mode"] = df["Contact_Mode"].replace(value_mapping)

df["Contact_Mode"].unique()

array(['Email', 'SMS', 'Social media apps', 'Phone', 'In person', 'Fax'],
      dtype=object)

In [7]:
# Complainant_Gender: Extract male and female
df = df[df["Complainant_Gender"].isin(["Male", "Female"])]
df["Complainant_Gender"].unique()

array(['Male', 'Female'], dtype=object)

In [8]:
# Category_Level_2 (Scam Type): Rename column and extract relevant row
df.rename(columns={'Category_Level_2': 'Scam_Type'}, inplace=True)

scam_types = ["Attempts to gain your personal information", "Dating and romance", "Investment scams",
               "Jobs and employment", "Buying or selling"]
df = df[df["Scam_Type"].isin(scam_types)]

# Replace values
value_mapping = {
    "Attempts to gain your personal information": "Phishing Scams",
    "Dating and romance": "Romance Scams",
    "Investment scams": "Investment Scams",
    "Jobs and employment": "Job Scams",
    "Buying or selling": "E-commerce Scams"
}

df["Scam_Type"] = df["Scam_Type"].replace(value_mapping)

df["Scam_Type"].unique()

array(['E-commerce Scams', 'Job Scams', 'Investment Scams',
       'Romance Scams', 'Phishing Scams'], dtype=object)

In [9]:
# Amount_lost: Rename column and cast to float
df.rename(columns={'Amount_lost': 'Amount_Lost'}, inplace=True)

df["Amount_Lost"] = df["Amount_Lost"].replace("[\$,]", "", regex=True).astype(float)

print(df["Amount_Lost"].dtype)
df.head()

float64


,StartOfMonth,Address_State,Contact_Mode,Complainant_Age,Complainant_Gender,Scam_Type,Category_Level_3,Amount_Lost,Number_of_reports,Year,Month
0,2025-01-01,Victoria,Email,65 and over,Male,E-commerce Scams,False billing,69.0,41,2025,1
1,2025-02-01,Victoria,Email,65 and over,Male,E-commerce Scams,False billing,0.0,29,2025,2
2,2025-03-01,Victoria,Email,65 and over,Male,E-commerce Scams,False billing,2643.0,39,2025,3
3,2025-04-01,Victoria,Email,65 and over,Male,E-commerce Scams,False billing,20000.0,30,2025,4
4,2025-05-01,Victoria,Email,65 and over,Male,E-commerce Scams,False billing,86416.0,28,2025,5


In [10]:
# Drop useless columns
df.drop(["Category_Level_3", "StartOfMonth"], axis=1, inplace=True)
df.head()

,Address_State,Contact_Mode,Complainant_Age,Complainant_Gender,Scam_Type,Amount_Lost,Number_of_reports,Year,Month
0,Victoria,Email,65 and over,Male,E-commerce Scams,69.0,41,2025,1
1,Victoria,Email,65 and over,Male,E-commerce Scams,0.0,29,2025,2
2,Victoria,Email,65 and over,Male,E-commerce Scams,2643.0,39,2025,3
3,Victoria,Email,65 and over,Male,E-commerce Scams,20000.0,30,2025,4
4,Victoria,Email,65 and over,Male,E-commerce Scams,86416.0,28,2025,5


In [11]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

# Database connection details
load_dotenv()

DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_PORT = os.getenv('DB_PORT')
DB_NAME = os.getenv('DB_NAME')

# Create the connection engine
engine = create_engine(
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
)

df.to_sql("data_insight", engine, if_exists="replace", index=False)

726